# SURFMAP batch — tutte le proteine, mappa completa + ROI25 interattiva

Questa versione applica **lo stesso identico metodo pixel-perfect della versione per singola proteina** a tutte le proteine presenti in una cartella `RUN`.

Struttura attesa:

```text
RUN/
├── proteina1/
│   └── ...smooth.../
│       └── file_smoothed_matrix.txt
├── proteina2/
│   └── ...smooth.../
│       └── file_smoothed_matrix.txt
└── proteina3/
    └── ...smooth.../
        └── file_smoothed_matrix.txt
```

Per non caricare contemporaneamente decine o centinaia di grafici interattivi, la visualizzazione usa un **menu a tendina**:

1. scegli una proteina;
2. viene caricata soltanto quella matrice;
3. vengono mostrate la **mappa completa** e la sua **ROI25**;
4. cambiando proteina vengono sostituite le due figure.

Il salvataggio batch è separato dalla visualizzazione e può creare automaticamente full map e ROI per tutte le proteine.


## 0. Dipendenze

Se necessario, esegui una sola volta:

```python
%pip install --upgrade pandas numpy matplotlib pillow plotly nbformat ipywidgets
```

Dopo l'installazione di `ipywidgets` può essere necessario riavviare il kernel.

In [6]:
#%pip install --upgrade pandas numpy matplotlib pillow plotly nbformat ipywidgets

In [7]:
from pathlib import Path
import html
import warnings

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np
import pandas as pd
from PIL import Image
import plotly.graph_objects as go

from IPython.display import display, clear_output

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

## 1. Percorsi e parametri

Modifica soltanto:

- `RUN_DIR`: cartella che contiene tutte le cartelle delle proteine;
- `OUTPUT_DIR`: cartella in cui vuoi salvare tutti gli output.

Il notebook cerca, dentro ogni cartella proteina, una cartella il cui nome contiene `smooth` e al suo interno il file `.txt`.

La geometria e la colorazione restano quelle della versione singola-proteina.

In [8]:
RUN_DIR = Path(
    r"/home/chiara/Desktop/COMENZE/Comenze_pipeline/Surface_analysis/surfmap/rad0.5_s1_nosmooth"
)

OUTPUT_BASE_DIR = Path(
    r"/home/chiara/Desktop/COMENZE/Comenze_pipeline/Surface_analysis/SURFMAP_VALIDATION"
)

# SURFMAP_VALIDATION/rad0.5_s1_nosmooth_map_and_ROI
OUTPUT_DIR = (
    OUTPUT_BASE_DIR
    / f"{RUN_DIR.name}_map_and_ROI_test"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MAP_HEIGHT = 180
MAP_WIDTH = 360
ROI_RADIUS = 50

# I valori NON vengono rinormalizzati.
VALUE_MIN = -1.0
VALUE_MAX = 1.0

COLORMAP = "bwr_r"

# Pixel realmente vuoti (NaN / Inf) = grigio.
EMPTY_RGBA = np.array(
    [128, 128, 128, 255],
    dtype=np.uint8,
)

# Fuori dal cerchio della ROI = bianco.
OUTSIDE_ROI_RGBA = np.array(
    [255, 255, 255, 255],
    dtype=np.uint8,
)

print("RUN:", RUN_DIR)
print("RUN esiste:", RUN_DIR.is_dir())

print("OUTPUT BASE:", OUTPUT_BASE_DIR)
print("OUTPUT:", OUTPUT_DIR)
print("OUTPUT esiste:", OUTPUT_DIR.is_dir())

print("Dimensione attesa:", MAP_HEIGHT, "x", MAP_WIDTH)
print("ROI: raggio", ROI_RADIUS, "pixel")

RUN: /home/chiara/Desktop/COMENZE/Comenze_pipeline/Surface_analysis/surfmap/rad0.5_s1_nosmooth
RUN esiste: True
OUTPUT BASE: /home/chiara/Desktop/COMENZE/Comenze_pipeline/Surface_analysis/SURFMAP_VALIDATION
OUTPUT: /home/chiara/Desktop/COMENZE/Comenze_pipeline/Surface_analysis/SURFMAP_VALIDATION/rad0.5_s1_nosmooth_map_and_ROI_test
OUTPUT esiste: True
Dimensione attesa: 180 x 360
ROI: raggio 50 pixel


## 2. Ricerca automatica delle smoothed matrices

Per ogni sottocartella immediata di `RUN_DIR`:

- il nome della cartella viene usato come nome della proteina;
- vengono cercate ricorsivamente cartelle contenenti `smooth` nel nome;
- viene preferito un file con `smoothed_matrix` nel nome;
- se esiste un solo `.txt`, viene usato quello;
- se ci sono più file ambigui, la proteina viene segnalata invece di scegliere un file arbitrariamente.

In [9]:
def discover_protein_matrices(run_dir):
    if not run_dir.is_dir():
        raise NotADirectoryError(
            f"RUN_DIR non esiste o non è una cartella: {run_dir}"
        )

    protein_files = {}
    rows = []

    protein_dirs = sorted(
        [
            path
            for path in run_dir.iterdir()
            if path.is_dir()
        ],
        key=lambda path: path.name.lower(),
    )

    for protein_dir in protein_dirs:
        smooth_dirs = sorted(
            [
                path
                for path in protein_dir.rglob("*")
                if path.is_dir()
                and "smooth" in path.name.lower()
            ],
            key=lambda path: str(path).lower(),
        )

        candidates = []

        for smooth_dir in smooth_dirs:
            candidates.extend(
                sorted(
                    smooth_dir.glob("*.txt"),
                    key=lambda path: path.name.lower(),
                )
            )

        # Rimuove eventuali duplicati mantenendo l'ordine.
        unique_candidates = []
        seen = set()

        for candidate in candidates:
            resolved = candidate.resolve()

            if resolved not in seen:
                seen.add(resolved)
                unique_candidates.append(candidate)

        preferred = [
            candidate
            for candidate in unique_candidates
            if "smoothed_matrix" in candidate.name.lower()
        ]

        chosen = None
        status = ""

        if len(preferred) == 1:
            chosen = preferred[0]
            status = "OK — smoothed_matrix"

        elif len(preferred) > 1:
            status = (
                "AMBIGUO — più file smoothed_matrix: "
                + " | ".join(
                    candidate.name
                    for candidate in preferred
                )
            )

        elif len(unique_candidates) == 1:
            chosen = unique_candidates[0]
            status = "OK — unico TXT"

        elif len(unique_candidates) == 0:
            if len(smooth_dirs) == 0:
                status = "NON TROVATA — nessuna cartella smooth"
            else:
                status = "NON TROVATA — nessun TXT nella cartella smooth"

        else:
            status = (
                "AMBIGUO — più TXT: "
                + " | ".join(
                    candidate.name
                    for candidate in unique_candidates
                )
            )

        if chosen is not None:
            protein_files[protein_dir.name] = chosen

        rows.append(
            {
                "protein": protein_dir.name,
                "matrix_file": (
                    str(chosen)
                    if chosen is not None
                    else ""
                ),
                "status": status,
            }
        )

    discovery_table = pd.DataFrame(rows)

    return protein_files, discovery_table


PROTEIN_FILES, discovery_table = discover_protein_matrices(
    RUN_DIR
)

print(
    f"Proteine utilizzabili: {len(PROTEIN_FILES)} "
    f"su {len(discovery_table)} cartelle trovate."
)

display(
    discovery_table
)

Proteine utilizzabili: 42 su 42 cartelle trovate.


,protein,matrix_file,status
0,output_SURFMAP_1CEX_cutfv_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK — smoothed_matrix
1,output_SURFMAP_1CRL_lipCr_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK — smoothed_matrix
2,output_SURFMAP_1CUS_fsc_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK — smoothed_matrix
3,output_SURFMAP_1PPN_pap_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK — smoothed_matrix
4,output_SURFMAP_1SBC_sub_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK — smoothed_matrix
5,output_SURFMAP_1TPO_tryp_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK — smoothed_matrix
6,output_SURFMAP_1WYC_NylB_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK — smoothed_matrix
7,output_SURFMAP_2CHA_chym_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK — smoothed_matrix
8,output_SURFMAP_3AXG_NylCa_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK — smoothed_matrix
9,output_SURFMAP_3GBS_aoc_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK — smoothed_matrix


## 3. Lettura della smoothed matrix

È la stessa funzione della versione singola-proteina.  
Non viene applicata alcuna normalizzazione dei valori.

In [10]:
def load_smoothed_matrix(file_path, expected_height=90, expected_width=180):
    if not file_path.is_file():
        raise FileNotFoundError(
            f"File smoothed matrix non trovato: {file_path}"
        )

    data = pd.read_csv(
        file_path,
        sep="\t",
        dtype=str,
    )

    data.columns = [
        str(column).strip()
        for column in data.columns
    ]

    column_lookup = {
        str(column).strip().lower(): column
        for column in data.columns
    }

    def find_column(candidates, required=True):
        for candidate in candidates:
            if candidate.lower() in column_lookup:
                return column_lookup[candidate.lower()]

        if required:
            raise ValueError(
                "Nel file manca una delle colonne richieste: "
                + ", ".join(candidates)
            )

        return None

    absc_column = find_column(["absc", "x"])
    ord_column = find_column(["ord", "y"])
    value_column = find_column(["svalue", "value"])

    residue_column = find_column(
        [
            "residue",
            "residues",
            "res",
            "residue_id",
            "residue_name",
        ],
        required=False,
    )

    working = pd.DataFrame({
        "absc": pd.to_numeric(
            data[absc_column],
            errors="coerce",
        ),
        "ord": pd.to_numeric(
            data[ord_column],
            errors="coerce",
        ),
        "svalue": pd.to_numeric(
            data[value_column],
            errors="coerce",
        ),
    })

    if residue_column is None:
        working["residue"] = ""
        warnings.warn(
            "Nel file non è stata trovata una colonna residue. "
            "Il tooltip mostrerà comunque coordinate e valore."
        )
    else:
        working["residue"] = (
            data[residue_column]
            .fillna("")
            .astype(str)
        )

    working = working.dropna(
        subset=["absc", "ord"]
    ).copy()

    duplicated_coordinates = working.duplicated(
        subset=["ord", "absc"],
        keep=False,
    )

    if duplicated_coordinates.any():
        duplicate_preview = (
            working.loc[
                duplicated_coordinates,
                ["ord", "absc"],
            ]
            .head(10)
            .to_dict("records")
        )

        raise ValueError(
            "Il file contiene coordinate ord/absc duplicate. "
            f"Esempi: {duplicate_preview}"
        )

    value_table = (
        working
        .pivot(
            index="ord",
            columns="absc",
            values="svalue",
        )
        .sort_index(axis=0)
        .sort_index(axis=1)
    )

    residue_table = (
        working
        .pivot(
            index="ord",
            columns="absc",
            values="residue",
        )
        .reindex(
            index=value_table.index,
            columns=value_table.columns,
        )
    )

    surface_map = value_table.to_numpy(
        dtype=float,
        copy=True,
    )

    surface_map[
        ~np.isfinite(surface_map)
    ] = np.nan

    residue_map = (
        residue_table
        .fillna("")
        .astype(str)
        .to_numpy(copy=True)
    )

    ord_values = value_table.index.to_numpy(copy=True)
    absc_values = value_table.columns.to_numpy(copy=True)

    expected_shape = (
        expected_height,
        expected_width,
    )

    if surface_map.shape != expected_shape:
        raise ValueError(
            f"Dimensione ricostruita: {surface_map.shape}. "
            f"Dimensione attesa: {expected_shape}. "
            "Controlla di avere selezionato il file "
            "*_smoothed_matrix.txt corretto."
        )

    # Il file Surfmap può contenere Inf e -Inf nelle zone esterne.
    # Questi valori vengono trasformati in NaN nella matrice e devono quindi
    # essere esclusi anche dal controllo di integrità.
    original_numeric = working["svalue"].to_numpy(
        dtype=float,
        copy=True,
    )

    original_finite = np.sort(
        original_numeric[
            np.isfinite(original_numeric)
        ]
    )

    reconstructed_finite = np.sort(
        surface_map[
            np.isfinite(surface_map)
        ]
    )

    if (
        original_finite.shape
        != reconstructed_finite.shape
        or not np.array_equal(
            original_finite,
            reconstructed_finite,
        )
    ):
        max_difference = np.nan

        if (
            original_finite.shape
            == reconstructed_finite.shape
            and original_finite.size > 0
        ):
            max_difference = float(
                np.max(
                    np.abs(
                        original_finite
                        - reconstructed_finite
                    )
                )
            )

        raise RuntimeError(
            "Controllo di integrità non superato. "
            f"Valori finiti originali: {original_finite.size}; "
            f"valori finiti ricostruiti: {reconstructed_finite.size}; "
            f"differenza massima: {max_difference}."
        )

    non_finite_count = int(
        np.sum(
            ~np.isfinite(original_numeric)
        )
    )

    print(
        "Controllo valori superato: "
        f"{original_finite.size} valori finiti invariati; "
        f"{non_finite_count} valori Inf/NaN esclusi dalla superficie."
    )

    return (
        working,
        surface_map,
        residue_map,
        absc_values,
        ord_values,
    )

## 4. Estrazione ROI centrale

Stesso ritaglio della versione singola-proteina:

- matrice `90 × 180`;
- centro `(45, 90)`;
- raggio `25`;
- patch `50 × 50`;
- maschera circolare in pixel.

In [11]:
def extract_central_roi(
    value_map,
    residue_map,
    absc_values,
    ord_values,
    radius=25,
):
    height, width = value_map.shape

    center_y = height // 2
    center_x = width // 2

    # Il raggio geometrico può essere decimale.
    radius_float = float(radius)

    # Per il ritaglio dell'array servono invece indici interi.
    crop_radius = int(
        np.ceil(radius_float)
    )

    y_start = max(
        0,
        center_y - crop_radius,
    )

    y_end = min(
        height,
        center_y + crop_radius,
    )

    x_start = max(
        0,
        center_x - crop_radius,
    )

    x_end = min(
        width,
        center_x + crop_radius,
    )


    roi_values = value_map[
        y_start:y_end,
        x_start:x_end,
    ].copy()

    roi_residues = residue_map[
        y_start:y_end,
        x_start:x_end,
    ].copy()

    roi_absc_values = absc_values[
        x_start:x_end
    ].copy()

    roi_ord_values = ord_values[
        y_start:y_end
    ].copy()


    roi_height, roi_width = (
        roi_values.shape
    )

    # Centro reale della mappa espresso
    # nelle coordinate locali della patch.
    local_center_y = (
        center_y - y_start
    )

    local_center_x = (
        center_x - x_start
    )


    yy, xx = np.ogrid[
        :roi_height,
        :roi_width,
    ]


    # IMPORTANTE:
    # qui usiamo il raggio DECIMALE reale,
    # non crop_radius.
    circle_mask = (
        (xx - local_center_x) ** 2
        + (yy - local_center_y) ** 2
    ) <= radius_float ** 2


    roi_values_masked = (
        roi_values.copy()
    )

    roi_values_masked[
        ~circle_mask
    ] = np.nan


    return {
        "values": roi_values,
        "values_masked": roi_values_masked,
        "residues": roi_residues,
        "mask": circle_mask,
        "x_start": x_start,
        "x_end": x_end,
        "y_start": y_start,
        "y_end": y_end,
        "center_x": center_x,
        "center_y": center_y,
        "absc_values": roi_absc_values,
        "ord_values": roi_ord_values,
        "radius": radius_float,
    }

## 5. Conversione pixel-perfect dei valori in colori

La logica resta invariata:

- `svalue` non viene modificato;
- scala colori `[-1, +1]`;
- colormap `bwr_r`;
- pixel reali `NaN/Inf` = **grigio**;
- area esterna al cerchio della ROI = **bianco**.

In [12]:
def values_to_rgba(
    values,
    residues=None,
    valid_mask=None,
    value_min=-1.0,
    value_max=1.0,
    colormap="bwr_r",
):
    values = np.asarray(
        values,
        dtype=float,
    )

    finite_mask = np.isfinite(
        values
    )

    normalization = Normalize(
        vmin=value_min,
        vmax=value_max,
        clip=True,
    )

    cmap = plt.get_cmap(
        colormap
    )

    # Non-finite values are temporarily mapped to 0
    # only to generate the RGBA image.
    safe_values = np.where(
        finite_mask,
        values,
        0.0,
    )

    rgba_float = cmap(
        normalization(
            safe_values
        )
    )

    rgba_uint8 = np.round(
        rgba_float * 255.0
    ).astype(
        np.uint8
    )


    # ========================================================
    # EMPTY PIXEL DEFINITION
    #
    # Empty ONLY when the value is non-finite:
    # NaN / Inf / -Inf.
    #
    # Residue information is NOT used to define empty pixels.
    # ========================================================

    empty_mask = (
        ~finite_mask
    )


    # ONLY true empty pixels = GRAY
    rgba_uint8[
        empty_mask
    ] = EMPTY_RGBA


    # Outside the circular ROI = WHITE
    if valid_mask is not None:

        rgba_uint8[
            ~np.asarray(
                valid_mask,
                dtype=bool,
            )
        ] = OUTSIDE_ROI_RGBA


    return rgba_uint8


## 6. Tooltip

Stesso tooltip pixel-per-pixel:

- riga e colonna;
- coordinate originali `ord` e `absc`;
- `svalue`;
- residuo.

In [13]:
def format_value(value):
    if np.isfinite(value):
        return f"{float(value):.8f}"
    return "NA"


def build_full_hover_text(
    values,
    residues,
    absc_values,
    ord_values,
):
    height, width = values.shape

    hover_text = np.empty(
        (height, width),
        dtype=object,
    )

    for row in range(height):
        for column in range(width):
            residue = str(
                residues[row, column]
            ).strip()

            if residue == "":
                residue = "non disponibile"

            hover_text[row, column] = (
                f"riga matrice: {row}<br>"
                f"colonna matrice: {column}<br>"
                f"ord originale: {ord_values[row]}<br>"
                f"absc originale: {absc_values[column]}<br>"
                f"svalue originale: "
                f"{format_value(values[row, column])}<br>"
                f"residuo: {residue}"
            )

    return hover_text


def build_roi_hover_text(
    roi_values,
    roi_residues,
    roi_mask,
    x_start,
    y_start,
    roi_absc_values,
    roi_ord_values,
):
    height, width = roi_values.shape

    hover_text = np.empty(
        (height, width),
        dtype=object,
    )

    for local_row in range(height):
        for local_column in range(width):
            if not roi_mask[
                local_row,
                local_column,
            ]:
                hover_text[
                    local_row,
                    local_column,
                ] = (
                    "fuori dalla ROI circolare"
                )
                continue

            global_row = (
                y_start
                + local_row
            )

            global_column = (
                x_start
                + local_column
            )

            residue = str(
                roi_residues[
                    local_row,
                    local_column,
                ]
            ).strip()

            if residue == "":
                residue = "non disponibile"

            hover_text[
                local_row,
                local_column,
            ] = (
                f"riga ROI: {local_row}<br>"
                f"colonna ROI: {local_column}<br>"
                f"riga matrice: {global_row}<br>"
                f"colonna matrice: {global_column}<br>"
                f"ord originale: "
                f"{roi_ord_values[local_row]}<br>"
                f"absc originale: "
                f"{roi_absc_values[local_column]}<br>"
                f"svalue originale: "
                f"{format_value(roi_values[local_row, local_column])}<br>"
                f"residuo: {residue}"
            )

    return hover_text

## 7. Visualizzazione interattiva pixel-perfect

Viene mantenuto `go.Image`: nessun `scatter`, nessuna interpolazione e nessuna modifica delle patch.

In [14]:
def make_interactive_image(
    rgba_image,
    hover_text,
    width,
    height,
):
    image_height, image_width = rgba_image.shape[:2]

    figure = go.Figure(
        go.Image(
            z=rgba_image,
            text=hover_text,
            hovertemplate=(
                "%{text}"
                "<extra></extra>"
            ),
            hoverinfo="text",
        )
    )

    figure.update_layout(
        width=width,
        height=height,
        margin=dict(
            l=0,
            r=0,
            t=0,
            b=0,
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        dragmode="pan",
    )

    figure.update_xaxes(
        visible=False,
        range=[
            -0.5,
            image_width - 0.5,
        ],
        constrain="domain",
        fixedrange=False,
    )

    figure.update_yaxes(
        visible=False,
        range=[
            image_height - 0.5,
            -0.5,
        ],
        scaleanchor="x",
        scaleratio=1,
        constrain="domain",
        fixedrange=False,
    )

    return figure


def show_interactive(figure):
    configuration = {
        "scrollZoom": True,
        "displaylogo": False,
        "showAxisDragHandles": False,
        "toImageButtonOptions": {
            "format": "png",
            "scale": 4,
        },
    }

    try:
        figure.show(
            config=configuration
        )

    except ValueError as exc:
        if "nbformat" in str(exc):
            print(
                "Visualizzazione interna non disponibile. "
                "Apro il grafico nel browser."
            )

            figure.show(
                renderer="browser",
                config=configuration,
            )
        else:
            raise

## 8. Elaborazione di una proteina

Questa funzione mette insieme le stesse operazioni della versione singola-proteina.

Non salva nulla: prepara soltanto full map, ROI, tooltip e figure interattive.

In [15]:
def process_protein(protein_name):
    if protein_name not in PROTEIN_FILES:
        raise KeyError(
            f"Proteina non trovata: {protein_name}"
        )

    matrix_file = PROTEIN_FILES[
        protein_name
    ]

    (
        matrix_data,
        surface_map,
        residue_map,
        absc_values,
        ord_values,
    ) = load_smoothed_matrix(
        matrix_file,
        expected_height=MAP_HEIGHT,
        expected_width=MAP_WIDTH,
    )

    roi = extract_central_roi(
        surface_map,
        residue_map,
        absc_values,
        ord_values,
        radius=ROI_RADIUS,
    )

    full_rgba = values_to_rgba(
        surface_map,
        residues=residue_map,
        value_min=VALUE_MIN,
        value_max=VALUE_MAX,
        colormap=COLORMAP,
    )

    roi_rgba = values_to_rgba(
        roi["values"],
        residues=roi["residues"],
        valid_mask=roi["mask"],
        value_min=VALUE_MIN,
        value_max=VALUE_MAX,
        colormap=COLORMAP,
    )

    full_hover_text = build_full_hover_text(
        surface_map,
        residue_map,
        absc_values,
        ord_values,
    )

    roi_hover_text = build_roi_hover_text(
        roi["values"],
        roi["residues"],
        roi["mask"],
        roi["x_start"],
        roi["y_start"],
        roi["absc_values"],
        roi["ord_values"],
    )

    full_plot = make_interactive_image(
        full_rgba,
        full_hover_text,
        width=1200,
        height=600,
    )

    roi_plot = make_interactive_image(
        roi_rgba,
        roi_hover_text,
        width=700,
        height=700,
    )

    finite_values = surface_map[
        np.isfinite(surface_map)
    ]

    return {
        "protein": protein_name,
        "matrix_file": matrix_file,
        "matrix_data": matrix_data,
        "surface_map": surface_map,
        "residue_map": residue_map,
        "absc_values": absc_values,
        "ord_values": ord_values,
        "roi": roi,
        "full_rgba": full_rgba,
        "roi_rgba": roi_rgba,
        "full_hover_text": full_hover_text,
        "roi_hover_text": roi_hover_text,
        "full_plot": full_plot,
        "roi_plot": roi_plot,
        "finite_min": (
            float(np.min(finite_values))
            if finite_values.size
            else np.nan
        ),
        "finite_max": (
            float(np.max(finite_values))
            if finite_values.size
            else np.nan
        ),
    }

## 9. Visualizzazione manuale di una singola proteina

Puoi sempre usare questa funzione senza menu a tendina:

```python
show_protein("nome_proteina")
```

Mostra prima la full map e subito sotto la ROI25.

In [16]:
def show_protein(protein_name):
    result = process_protein(
        protein_name
    )

    print("Proteina:", result["protein"])
    print("File:", result["matrix_file"])
    print(
        "Intervallo reale svalue:",
        result["finite_min"],
        "—",
        result["finite_max"],
    )

    print("\nMAPPA COMPLETA")
    show_interactive(
        result["full_plot"]
    )

    print("\nROI25")
    show_interactive(
        result["roi_plot"]
    )

    return result

## 10. Menu a tendina: scegli la proteina da visualizzare

Questa è la modalità consigliata.

Il notebook mostra **una sola proteina alla volta**:

- full map interattiva;
- ROI25 interattiva ingrandita.

Quando scegli un'altra proteina, le figure precedenti vengono sostituite.

In [ ]:
if widgets is None:
    print(
        "ipywidgets non è installato. "
        "Usa show_protein('nome_proteina') oppure installa ipywidgets."
    )

elif len(PROTEIN_FILES) == 0:
    print(
        "Nessuna proteina utilizzabile trovata. "
        "Controlla RUN_DIR e la tabella di ricerca."
    )

else:
    protein_dropdown = widgets.Dropdown(
        options=sorted(
            PROTEIN_FILES.keys(),
            key=str.lower,
        ),
        description="Proteina:",
        layout=widgets.Layout(
            width="700px",
        ),
        style={
            "description_width": "100px",
        },
    )

    protein_output = widgets.Output()

    def update_protein_view(change=None):
        protein_name = protein_dropdown.value

        with protein_output:
            clear_output(
                wait=True
            )

            try:
                result = process_protein(
                    protein_name
                )

                print(
                    "Proteina:",
                    result["protein"],
                )
                print(
                    "Matrice:",
                    result["matrix_file"],
                )
                print(
                    "svalue:",
                    result["finite_min"],
                    "—",
                    result["finite_max"],
                )

                print("\nMAPPA COMPLETA")
                show_interactive(
                    result["full_plot"]
                )

                print("\nROI25")
                show_interactive(
                    result["roi_plot"]
                )

            except Exception as exc:
                print(
                    f"Errore per {protein_name}: {exc}"
                )

    protein_dropdown.observe(
        update_protein_view,
        names="value",
    )

    display(
        widgets.VBox(
            [
                protein_dropdown,
                protein_output,
            ]
        )
    )

    update_protein_view()

## 11. Funzione di salvataggio per una proteina

Per ogni proteina vengono prodotti gli stessi file della versione singola:

- ROI esatta `50 × 50`;
- ROI ingrandita nearest-neighbor;
- full map esatta `180 × 90`;
- full map ingrandita nearest-neighbor.

Tutti i file vengono salvati in `OUTPUT_DIR` con il nome della proteina nel prefisso.

In [18]:
def safe_filename(text):
    return "".join(
        character
        if character.isalnum()
        or character in "-_."
        else "_"
        for character in str(text)
    )


def save_protein_pngs(
    protein_name,
    result=None,
):
    if result is None:
        result = process_protein(
            protein_name
        )

    protein_prefix = safe_filename(
        protein_name
    )

    matrix_prefix = safe_filename(
        result["matrix_file"].stem
    )

    prefix = (
        f"{protein_prefix}__"
        f"{matrix_prefix}"
    )

    roi_exact_path = (
        OUTPUT_DIR
        / f"{prefix}_ROI25_exact_50x50.png"
    )

    roi_large_path = (
        OUTPUT_DIR
        / f"{prefix}_ROI25_enlarged.png"
    )

    full_exact_path = (
        OUTPUT_DIR
        / f"{prefix}_full_map_exact_180x90.png"
    )

    full_large_path = (
        OUTPUT_DIR
        / f"{prefix}_full_map_enlarged.png"
    )

    roi_image = Image.fromarray(
        result["roi_rgba"],
        mode="RGBA",
    )

    roi_image.save(
        roi_exact_path,
        format="PNG",
    )

    ROI_ENLARGEMENT = 12

    roi_image.resize(
        (
            roi_image.width
            * ROI_ENLARGEMENT,
            roi_image.height
            * ROI_ENLARGEMENT,
        ),
        resample=Image.Resampling.NEAREST,
    ).save(
        roi_large_path,
        format="PNG",
    )

    full_image = Image.fromarray(
        result["full_rgba"],
        mode="RGBA",
    )

    full_image.save(
        full_exact_path,
        format="PNG",
    )

    FULL_MAP_ENLARGEMENT = 6

    full_image.resize(
        (
            full_image.width
            * FULL_MAP_ENLARGEMENT,
            full_image.height
            * FULL_MAP_ENLARGEMENT,
        ),
        resample=Image.Resampling.NEAREST,
    ).save(
        full_large_path,
        format="PNG",
    )

    return {
        "protein": protein_name,
        "matrix_file": str(
            result["matrix_file"]
        ),
        "roi_exact": str(
            roi_exact_path
        ),
        "roi_enlarged": str(
            roi_large_path
        ),
        "full_exact": str(
            full_exact_path
        ),
        "full_enlarged": str(
            full_large_path
        ),
    }

## 12. Salvataggio batch PNG di tutte le proteine

Esegui questa cella quando vuoi salvare full map e ROI di **tutte** le proteine trovate.

Un errore su una proteina non interrompe l'intero loop: viene registrato nella tabella finale.

In [19]:
batch_save_rows = []

for index, protein_name in enumerate(
    sorted(
        PROTEIN_FILES.keys(),
        key=str.lower,
    ),
    start=1,
):
    print(
        f"[{index}/{len(PROTEIN_FILES)}] "
        f"{protein_name}"
    )

    try:
        result = process_protein(
            protein_name
        )

        saved = save_protein_pngs(
            protein_name,
            result=result,
        )

        saved["status"] = "OK"

        batch_save_rows.append(
            saved
        )

    except Exception as exc:
        batch_save_rows.append(
            {
                "protein": protein_name,
                "matrix_file": str(
                    PROTEIN_FILES[
                        protein_name
                    ]
                ),
                "roi_exact": "",
                "roi_enlarged": "",
                "full_exact": "",
                "full_enlarged": "",
                "status": (
                    f"ERRORE — {exc}"
                ),
            }
        )

batch_save_table = pd.DataFrame(
    batch_save_rows
)

display(
    batch_save_table
)

print(
    "\nOutput salvati in:",
    OUTPUT_DIR.resolve(),
)

[1/42] output_SURFMAP_1CEX_cutfv_4rev_aligned_bfactor
Controllo valori superato: 40716 valori finiti invariati; 24084 valori Inf/NaN esclusi dalla superficie.
[2/42] output_SURFMAP_1CRL_lipCr_4rev_aligned_bfactor
Controllo valori superato: 40716 valori finiti invariati; 24084 valori Inf/NaN esclusi dalla superficie.
[3/42] output_SURFMAP_1CUS_fsc_4rev_aligned_bfactor
Controllo valori superato: 40716 valori finiti invariati; 24084 valori Inf/NaN esclusi dalla superficie.
[4/42] output_SURFMAP_1PPN_pap_4rev_aligned_bfactor
Controllo valori superato: 40716 valori finiti invariati; 24084 valori Inf/NaN esclusi dalla superficie.
[5/42] output_SURFMAP_1SBC_sub_4rev_aligned_bfactor
Controllo valori superato: 40716 valori finiti invariati; 24084 valori Inf/NaN esclusi dalla superficie.
[6/42] output_SURFMAP_1TPO_tryp_4rev_aligned_bfactor
Controllo valori superato: 40716 valori finiti invariati; 24084 valori Inf/NaN esclusi dalla superficie.
[7/42] output_SURFMAP_1WYC_NylB_4rev_aligned_bfactor


,protein,matrix_file,roi_exact,roi_enlarged,full_exact,full_enlarged,status
0,output_SURFMAP_1CEX_cutfv_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK
1,output_SURFMAP_1CRL_lipCr_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK
2,output_SURFMAP_1CUS_fsc_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK
3,output_SURFMAP_1PPN_pap_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK
4,output_SURFMAP_1SBC_sub_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK
5,output_SURFMAP_1TPO_tryp_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK
6,output_SURFMAP_1WYC_NylB_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK
7,output_SURFMAP_2CHA_chym_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK
8,output_SURFMAP_3AXG_NylCa_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK
9,output_SURFMAP_3GBS_aoc_4rev_aligned_bfactor,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,OK



Output salvati in: /home/chiara/Desktop/COMENZE/Comenze_pipeline/Surface_analysis/SURFMAP_VALIDATION/rad0.5_s1_nosmooth_map_and_ROI_test


## 13. Empty pixels calculation

In [20]:
# ============================================================
# EMPTY PIXEL STATISTICS — SMOOTHED MATRICES
# ============================================================

empty_roi_rows = []


# ============================================================
# CHECK THAT THE MATRICES BELONG TO THE CURRENT RUN
# ============================================================

current_run_path = RUN_DIR.resolve()

wrong_paths = []

for protein_name, matrix_path in PROTEIN_FILES.items():

    matrix_path = Path(
        matrix_path
    ).resolve()

    if current_run_path not in matrix_path.parents:

        wrong_paths.append(
            {
                "protein": protein_name,
                "matrix_path": matrix_path,
            }
        )


if wrong_paths:

    print("=" * 90)
    print("ERROR: MATRIX PATHS DO NOT MATCH THE CURRENT RUN")
    print("=" * 90)

    print()
    print("Current RUN:")
    print(current_run_path)

    print()
    print("Example matrix currently stored in PROTEIN_FILES:")
    print(
        wrong_paths[0]["matrix_path"]
    )

    raise RuntimeError(
        "PROTEIN_FILES still contains matrices from another RUN. "
        "Re-run the smoothed-matrix discovery cell before running "
        "the statistics."
    )


print("=" * 90)
print("EMPTY PIXEL ANALYSIS — SMOOTHED MATRICES")
print("=" * 90)

print(
    f"Current RUN: {RUN_DIR.name}"
)

print(
    f"Smoothed matrices found: "
    f"{len(PROTEIN_FILES)}"
)

print()


# ============================================================
# ANALYSE EVERY PROTEIN
# ============================================================

for protein_name in sorted(
    PROTEIN_FILES.keys(),
    key=str.lower,
):

    try:

        # Uses the SAME smoothed matrix and SAME ROI
        # already used for the interactive maps
        result = process_protein(
            protein_name
        )


        roi_values = result[
            "roi"
        ][
            "values"
        ]

        roi_mask = result[
            "roi"
        ][
            "mask"
        ]

        roi_residues = result[
            "roi"
        ][
            "residues"
        ]


# ----------------------------------------------------
# EMPTY PIXELS
#
# Empty ONLY when the value is:
# NaN / Inf / -Inf.
#
# value == 0.0 is VALID and is NOT counted as empty.
# Residue information is NOT used to define empty pixels.
# ----------------------------------------------------

        finite_mask = np.isfinite(
            roi_values
        )

        empty_mask = (
            ~finite_mask
        )

        # Only pixels INSIDE the circular ROI
        total_roi_pixels = int(
            np.count_nonzero(
                roi_mask
            )
        )


        empty_roi_pixels = int(
            np.count_nonzero(
                empty_mask
                & roi_mask
            )
        )


        valid_roi_pixels = (
            total_roi_pixels
            - empty_roi_pixels
        )


        empty_percentage = (
            empty_roi_pixels
            / total_roi_pixels
            * 100.0
            if total_roi_pixels > 0
            else np.nan
        )


        matrix_path = Path(
            result["matrix_file"]
        ).resolve()


        empty_roi_rows.append(
            {
                "Protein": protein_name,
                "RUN": RUN_DIR.name,
                "Smoothed_matrix_file": str(
                    matrix_path
                ),
                "Total_ROI_pixels": total_roi_pixels,
                "Valid_ROI_pixels": valid_roi_pixels,
                "Empty_ROI_pixels": empty_roi_pixels,
                "Empty_ROI_percent": empty_percentage,
            }
        )


        # ----------------------------------------------------
        # PRINT RESULTS FOR EACH PROTEIN
        # ----------------------------------------------------

        print(
            f"Protein: {protein_name}"
        )

        print(
            f"  Smoothed matrix   : "
            f"{matrix_path}"
        )

        print(
            f"  Total ROI pixels  : "
            f"{total_roi_pixels}"
        )

        print(
            f"  Valid ROI pixels  : "
            f"{valid_roi_pixels}"
        )

        print(
            f"  Empty ROI pixels  : "
            f"{empty_roi_pixels}"
        )

        print(
            f"  Empty ROI percent : "
            f"{empty_percentage:.2f}%"
        )

        print(
            "-" * 90
        )


    except Exception as exc:

        print(
            f"ERROR for {protein_name}: "
            f"{exc}"
        )

        print(
            "-" * 90
        )


# ============================================================
# CREATE RESULTS TABLE
# ============================================================

empty_roi_table = pd.DataFrame(
    empty_roi_rows
)


if empty_roi_table.empty:

    raise RuntimeError(
        "No protein was successfully analysed."
    )


empty_percentages = (
    empty_roi_table[
        "Empty_ROI_percent"
    ]
    .dropna()
)


# ============================================================
# SUMMARY STATISTICS
# ============================================================

mean_empty_percentage = (
    empty_percentages.mean()
)

median_empty_percentage = (
    empty_percentages.median()
)

percentile_25 = (
    empty_percentages.quantile(
        0.25
    )
)

percentile_75 = (
    empty_percentages.quantile(
        0.75
    )
)

percentile_90 = (
    empty_percentages.quantile(
        0.90
    )
)


# ============================================================
# PRINT SUMMARY BEFORE SAVING
# ============================================================

print()
print("=" * 90)
print("SUMMARY STATISTICS")
print("=" * 90)

print(
    f"RUN: "
    f"{RUN_DIR.name}"
)

print(
    f"Number of proteins analysed: "
    f"{len(empty_roi_table)}"
)

print()

print(
    f"Mean empty ROI pixels: "
    f"{mean_empty_percentage:.2f}%"
)

print(
    f"Median empty ROI pixels: "
    f"{median_empty_percentage:.2f}%"
)

print(
    f"25th percentile: "
    f"{percentile_25:.2f}%"
)

print(
    f"75th percentile: "
    f"{percentile_75:.2f}%"
)

print(
    f"25th–75th percentile range: "
    f"{percentile_25:.2f}% – "
    f"{percentile_75:.2f}%"
)

print(
    f"90th percentile: "
    f"{percentile_90:.2f}%"
)

print()
print(
    "Complete results table:"
)

display(
    empty_roi_table
)


# ============================================================
# ADD SUMMARY ROWS TO CSV
# ============================================================

summary_rows = pd.DataFrame(
    [
        {
            "Protein": "MEAN_ALL_PROTEINS",
            "RUN": RUN_DIR.name,
            "Smoothed_matrix_file": "",
            "Total_ROI_pixels": "",
            "Valid_ROI_pixels": "",
            "Empty_ROI_pixels": "",
            "Empty_ROI_percent": mean_empty_percentage,
        },

        {
            "Protein": "MEDIAN_ALL_PROTEINS",
            "RUN": RUN_DIR.name,
            "Smoothed_matrix_file": "",
            "Total_ROI_pixels": "",
            "Valid_ROI_pixels": "",
            "Empty_ROI_pixels": "",
            "Empty_ROI_percent": median_empty_percentage,
        },

        {
            "Protein": "25TH_PERCENTILE",
            "RUN": RUN_DIR.name,
            "Smoothed_matrix_file": "",
            "Total_ROI_pixels": "",
            "Valid_ROI_pixels": "",
            "Empty_ROI_pixels": "",
            "Empty_ROI_percent": percentile_25,
        },

        {
            "Protein": "75TH_PERCENTILE",
            "RUN": RUN_DIR.name,
            "Smoothed_matrix_file": "",
            "Total_ROI_pixels": "",
            "Valid_ROI_pixels": "",
            "Empty_ROI_pixels": "",
            "Empty_ROI_percent": percentile_75,
        },

        {
            "Protein": "90TH_PERCENTILE",
            "RUN": RUN_DIR.name,
            "Smoothed_matrix_file": "",
            "Total_ROI_pixels": "",
            "Valid_ROI_pixels": "",
            "Empty_ROI_pixels": "",
            "Empty_ROI_percent": percentile_90,
        },
    ]
)


final_empty_roi_table = pd.concat(
    [
        empty_roi_table,
        summary_rows,
    ],
    ignore_index=True,
)


# ============================================================
# SAVE CSV
# ============================================================

# Uses the same folder where maps and ROIs are saved.
stats_output_dir = (
    SAVE_DIR
    if "SAVE_DIR" in globals()
    else OUTPUT_DIR
)


csv_path = (
    stats_output_dir
    / f"{RUN_DIR.name}_ROI25_empty_pixels_statistics.csv"
)


final_empty_roi_table.to_csv(
    csv_path,
    index=False,
)


print()
print("=" * 90)
print("CSV FILE SAVED")
print("=" * 90)

print(
    f"CSV saved to: "
    f"{csv_path.resolve()}"
)

EMPTY PIXEL ANALYSIS — SMOOTHED MATRICES
Current RUN: rad0.5_s1_nosmooth
Smoothed matrices found: 42

Controllo valori superato: 40716 valori finiti invariati; 24084 valori Inf/NaN esclusi dalla superficie.
Protein: output_SURFMAP_1CEX_cutfv_4rev_aligned_bfactor
  Smoothed matrix   : /home/chiara/Desktop/COMENZE/Comenze_pipeline/Surface_analysis/surfmap/rad0.5_s1_nosmooth/output_SURFMAP_1CEX_cutfv_4rev_aligned_bfactor/smoothed_matrices/1CEX_cutfv_4rev_aligned_bfactor_smoothed_matrix.txt
  Total ROI pixels  : 7843
  Valid ROI pixels  : 7843
  Empty ROI pixels  : 0
  Empty ROI percent : 0.00%
------------------------------------------------------------------------------------------
Controllo valori superato: 40716 valori finiti invariati; 24084 valori Inf/NaN esclusi dalla superficie.
Protein: output_SURFMAP_1CRL_lipCr_4rev_aligned_bfactor
  Smoothed matrix   : /home/chiara/Desktop/COMENZE/Comenze_pipeline/Surface_analysis/surfmap/rad0.5_s1_nosmooth/output_SURFMAP_1CRL_lipCr_4rev_aligned

,Protein,RUN,Smoothed_matrix_file,Total_ROI_pixels,Valid_ROI_pixels,Empty_ROI_pixels,Empty_ROI_percent
0,output_SURFMAP_1CEX_cutfv_4rev_aligned_bfactor,rad0.5_s1_nosmooth,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,7843,7843,0,0.0
1,output_SURFMAP_1CRL_lipCr_4rev_aligned_bfactor,rad0.5_s1_nosmooth,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,7843,7843,0,0.0
2,output_SURFMAP_1CUS_fsc_4rev_aligned_bfactor,rad0.5_s1_nosmooth,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,7843,7843,0,0.0
3,output_SURFMAP_1PPN_pap_4rev_aligned_bfactor,rad0.5_s1_nosmooth,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,7843,7843,0,0.0
4,output_SURFMAP_1SBC_sub_4rev_aligned_bfactor,rad0.5_s1_nosmooth,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,7843,7843,0,0.0
5,output_SURFMAP_1TPO_tryp_4rev_aligned_bfactor,rad0.5_s1_nosmooth,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,7843,7843,0,0.0
6,output_SURFMAP_1WYC_NylB_4rev_aligned_bfactor,rad0.5_s1_nosmooth,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,7843,7843,0,0.0
7,output_SURFMAP_2CHA_chym_4rev_aligned_bfactor,rad0.5_s1_nosmooth,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,7843,7843,0,0.0
8,output_SURFMAP_3AXG_NylCa_4rev_aligned_bfactor,rad0.5_s1_nosmooth,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,7843,7843,0,0.0
9,output_SURFMAP_3GBS_aoc_4rev_aligned_bfactor,rad0.5_s1_nosmooth,/home/chiara/Desktop/COMENZE/Comenze_pipeline/...,7843,7843,0,0.0



CSV FILE SAVED
CSV saved to: /home/chiara/Desktop/COMENZE/Comenze_pipeline/Surface_analysis/SURFMAP_VALIDATION/rad0.5_s1_nosmooth_map_and_ROI_test/rad0.5_s1_nosmooth_ROI25_empty_pixels_statistics.csv


## 14. Salvataggio opzionale delle mappe interattive HTML

Esegui questa cella **solo se vuoi anche una versione interattiva su disco per ogni proteina**.

Per ogni proteina salva:

- full map interattiva `.html`;
- ROI25 interattiva `.html`.

Viene inoltre creato un file `SURFMAP_interactive_index.html` con una lista cliccabile di tutte le proteine.

In [21]:
html_rows = []

for index, protein_name in enumerate(
    sorted(
        PROTEIN_FILES.keys(),
        key=str.lower,
    ),
    start=1,
):
    print(
        f"[{index}/{len(PROTEIN_FILES)}] "
        f"{protein_name}"
    )

    try:
        result = process_protein(
            protein_name
        )

        protein_prefix = safe_filename(
            protein_name
        )

        matrix_prefix = safe_filename(
            result["matrix_file"].stem
        )

        prefix = (
            f"{protein_prefix}__"
            f"{matrix_prefix}"
        )

        full_html_path = (
            OUTPUT_DIR
            / f"{prefix}_full_map_interactive.html"
        )

        roi_html_path = (
            OUTPUT_DIR
            / f"{prefix}_ROI25_interactive.html"
        )

        result["full_plot"].write_html(
            full_html_path,
            include_plotlyjs=True,
            full_html=True,
        )

        result["roi_plot"].write_html(
            roi_html_path,
            include_plotlyjs=True,
            full_html=True,
        )

        html_rows.append(
            {
                "protein": protein_name,
                "full_html": full_html_path.name,
                "roi_html": roi_html_path.name,
                "status": "OK",
            }
        )

    except Exception as exc:
        html_rows.append(
            {
                "protein": protein_name,
                "full_html": "",
                "roi_html": "",
                "status": (
                    f"ERRORE — {exc}"
                ),
            }
        )

html_table = pd.DataFrame(
    html_rows
)

index_path = (
    OUTPUT_DIR
    / "SURFMAP_interactive_index.html"
)

index_lines = [
    "<!doctype html>",
    "<html>",
    "<head>",
    '<meta charset="utf-8">',
    "<title>SURFMAP interactive maps</title>",
    "</head>",
    "<body>",
    "<h1>SURFMAP interactive maps</h1>",
    "<ul>",
]

for row in html_rows:
    protein = html.escape(
        str(row["protein"])
    )

    if row["status"] == "OK":
        full_link = html.escape(
            row["full_html"]
        )

        roi_link = html.escape(
            row["roi_html"]
        )

        index_lines.append(
            "<li>"
            f"<strong>{protein}</strong> — "
            f'<a href="{full_link}">Full map</a> | '
            f'<a href="{roi_link}">ROI25</a>'
            "</li>"
        )
    else:
        status = html.escape(
            row["status"]
        )

        index_lines.append(
            "<li>"
            f"<strong>{protein}</strong> — "
            f"{status}"
            "</li>"
        )

index_lines.extend(
    [
        "</ul>",
        "</body>",
        "</html>",
    ]
)

index_path.write_text(
    "\n".join(index_lines),
    encoding="utf-8",
)

display(
    html_table
)

print(
    "Indice cliccabile creato:",
    index_path.resolve(),
)

[1/42] output_SURFMAP_1CEX_cutfv_4rev_aligned_bfactor
Controllo valori superato: 40716 valori finiti invariati; 24084 valori Inf/NaN esclusi dalla superficie.
[2/42] output_SURFMAP_1CRL_lipCr_4rev_aligned_bfactor
Controllo valori superato: 40716 valori finiti invariati; 24084 valori Inf/NaN esclusi dalla superficie.
[3/42] output_SURFMAP_1CUS_fsc_4rev_aligned_bfactor
Controllo valori superato: 40716 valori finiti invariati; 24084 valori Inf/NaN esclusi dalla superficie.
[4/42] output_SURFMAP_1PPN_pap_4rev_aligned_bfactor
Controllo valori superato: 40716 valori finiti invariati; 24084 valori Inf/NaN esclusi dalla superficie.
[5/42] output_SURFMAP_1SBC_sub_4rev_aligned_bfactor
Controllo valori superato: 40716 valori finiti invariati; 24084 valori Inf/NaN esclusi dalla superficie.
[6/42] output_SURFMAP_1TPO_tryp_4rev_aligned_bfactor
Controllo valori superato: 40716 valori finiti invariati; 24084 valori Inf/NaN esclusi dalla superficie.
[7/42] output_SURFMAP_1WYC_NylB_4rev_aligned_bfactor


,protein,full_html,roi_html,status
0,output_SURFMAP_1CEX_cutfv_4rev_aligned_bfactor,output_SURFMAP_1CEX_cutfv_4rev_aligned_bfactor...,output_SURFMAP_1CEX_cutfv_4rev_aligned_bfactor...,OK
1,output_SURFMAP_1CRL_lipCr_4rev_aligned_bfactor,output_SURFMAP_1CRL_lipCr_4rev_aligned_bfactor...,output_SURFMAP_1CRL_lipCr_4rev_aligned_bfactor...,OK
2,output_SURFMAP_1CUS_fsc_4rev_aligned_bfactor,output_SURFMAP_1CUS_fsc_4rev_aligned_bfactor__...,output_SURFMAP_1CUS_fsc_4rev_aligned_bfactor__...,OK
3,output_SURFMAP_1PPN_pap_4rev_aligned_bfactor,output_SURFMAP_1PPN_pap_4rev_aligned_bfactor__...,output_SURFMAP_1PPN_pap_4rev_aligned_bfactor__...,OK
4,output_SURFMAP_1SBC_sub_4rev_aligned_bfactor,output_SURFMAP_1SBC_sub_4rev_aligned_bfactor__...,output_SURFMAP_1SBC_sub_4rev_aligned_bfactor__...,OK
5,output_SURFMAP_1TPO_tryp_4rev_aligned_bfactor,output_SURFMAP_1TPO_tryp_4rev_aligned_bfactor_...,output_SURFMAP_1TPO_tryp_4rev_aligned_bfactor_...,OK
6,output_SURFMAP_1WYC_NylB_4rev_aligned_bfactor,output_SURFMAP_1WYC_NylB_4rev_aligned_bfactor_...,output_SURFMAP_1WYC_NylB_4rev_aligned_bfactor_...,OK
7,output_SURFMAP_2CHA_chym_4rev_aligned_bfactor,output_SURFMAP_2CHA_chym_4rev_aligned_bfactor_...,output_SURFMAP_2CHA_chym_4rev_aligned_bfactor_...,OK
8,output_SURFMAP_3AXG_NylCa_4rev_aligned_bfactor,output_SURFMAP_3AXG_NylCa_4rev_aligned_bfactor...,output_SURFMAP_3AXG_NylCa_4rev_aligned_bfactor...,OK
9,output_SURFMAP_3GBS_aoc_4rev_aligned_bfactor,output_SURFMAP_3GBS_aoc_4rev_aligned_bfactor__...,output_SURFMAP_3GBS_aoc_4rev_aligned_bfactor__...,OK


Indice cliccabile creato: /home/chiara/Desktop/COMENZE/Comenze_pipeline/Surface_analysis/SURFMAP_VALIDATION/rad0.5_s1_nosmooth_map_and_ROI_test/SURFMAP_interactive_index.html
